# Day 03 — baseline финансовой тональности

## D3-01 — ML runtime, config и проверенный вход

**Цель.** Подготовить окружение и проверить вход будущего обучения.
Вход — локальный processed CSV Financial PhraseBank `sentences_75agree` и
`reports/day02/provenance.json`; выход этого пункта — DataFrame `(N, 5)`,
typed config и подтверждённый runtime. Модель пока не создана и не обучена.

**Краткая теория.** `minimal-v1` уже выполнил детерминированную очистку в Day02.
`text_clean` — строки, а не числовые признаки. TF-IDF позже обучит vocabulary/IDF
на **train**, затем LogisticRegression обучит параметры классификатора.
`label_id` — category ID: `0=negative`, `1=neutral`, `2=positive`, не сила тональности.

Будущий поток:
`processed (N, 5) → group-aware split → train TF-IDF (N_train, V) → LogReg →
validation predictions (N_valid,) → metrics`, где `V <= 5000`.
D3-02 создаст split, D3-03 — estimators, D3-04 — fit/predict.
Baseline оценивается на validation; held-out test остаётся для D5-04.

Источник: takala/financial_phrasebank; Malo et al. (2014); CC-BY-NC-SA-3.0.
Notebook работает на реальных локальных данных и не выполняет download.
Если Day02 artifacts отсутствуют, отдельная команда из корня:
`PYTHONPATH=src venv/bin/python -m finnews_sentiment.data.prepare_day02`.
Она требует локальный источник; при его отсутствии сначала используйте
отдельную команду `prepare_phrasebank --download` из AGENTS.md.

Окружение: существующий `venv/`, pins — `requirements-day03.txt`.
Для воспроизведения окружения: `venv/bin/python -m pip install -r requirements-day03.txt`.
В этом checkout все нужные packages уже установлены.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.baseline_config import load_baseline_config
from finnews_sentiment.data.load_processed import load_processed_news
from finnews_sentiment.data.load_data import CLASS_ORDER
from finnews_sentiment.features.labels_duplicates import LABEL_MAPPING, analyze_duplicates

print(f"Interpreter: {sys.executable}")
print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Interpreter: /home/makarlistkov/projects/Text-classification/venv/bin/python
Project root: /home/makarlistkov/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Runtime и config

Импорты подтверждают доступность ML stack без создания estimators или fit.
Нужные версии закреплены отдельно от будущего большого curriculum stack.
Config содержит реальные пути, `text_clean`/`label_id`, seed=42, split fractions,
TF-IDF/LogReg/Dummy settings и относительные output directories.

`validation_size=0.2` — доля **outer-train**, поэтому номинально группы разделятся
примерно на 64%/16%/20%; split пока не выполняется. По строкам размеры могут
отличаться из-за групп дубликатов. L2 для установленного scikit-learn 1.9.1
задаётся `l1_ratio=0.0`, без deprecated `penalty`.
Параметры зафиксированы до измерения validation metrics.

In [2]:
from dataclasses import asdict
from importlib.metadata import version
import json
import platform

import pandas as pd
import sklearn
import scipy
import joblib
import threadpoolctl
from IPython.display import display

runtime_versions = {"python": platform.python_version()}
for package in ("pandas", "numpy", "scikit-learn", "scipy", "joblib", "threadpoolctl"):
    runtime_versions[package] = version(package)
display(pd.Series(runtime_versions, name="version").to_frame())

baseline_config = load_baseline_config(ROOT / "configs/train_baseline.json")
print(json.dumps(asdict(baseline_config), ensure_ascii=False, indent=2, default=str))

,version
python,3.14.6
pandas,3.0.6
numpy,2.5.3
scikit-learn,1.9.1
scipy,1.18.1
joblib,1.6.0
threadpoolctl,3.7.0


{
  "policy": "baseline-v1",
  "input_path": "data/processed/financial_phrasebank_75agree.csv",
  "provenance_path": "reports/day02/provenance.json",
  "text_col": "text_clean",
  "target_col": "label_id",
  "random_state": 42,
  "test_size": 0.2,
  "validation_size": 0.2,
  "tfidf": {
    "max_features": 5000,
    "ngram_range": [
      1,
      2
    ],
    "analyzer": "word",
    "lowercase": false,
    "stop_words": null,
    "min_df": 1,
    "max_df": 1.0,
    "use_idf": true,
    "smooth_idf": true,
    "sublinear_tf": false,
    "norm": "l2",
    "token_pattern": "(?u)\\b\\w\\w+\\b",
    "strip_accents": null
  },
  "logreg": {
    "solver": "lbfgs",
    "C": 1.0,
    "l1_ratio": 0.0,
    "max_iter": 1000,
    "tol": 0.0001,
    "class_weight": null,
    "random_state": 42
  },
  "dummy": {
    "strategy": "most_frequent"
  },
  "split_output_dir": "data/processed/day03",
  "report_output_dir": "reports/day03"
}


### Вызов loader: только проверка и чтение

Loader сверяет policy, source/revision, SHA-256 config/processed CSV, схему,
порядок и типы IDs, fixed mapping, наличие трёх классов и duplicate conflicts.
Он не вызывает preprocessing повторно, не удаляет строки/дубликаты и не делает fit.

Текстовые поля читаются с `keep_default_na=False`: буквальные `NA`/`nan` остаются
текстами. Integer поля сначала проверяются как CSV literals и только затем
преобразуются в `int64`, чтобы `1.5`, boolean и пропуски не маскировались.
Все необходимые ошибки останавливают подготовку; missing artifacts не скачиваются.

Проверяем повторную загрузку и неизменность bytes доступных входных файлов.
Checksums raw/interim здесь служат проверкой неизменности, а loader связывает
processed CSV с сохранённым provenance Day02, без повторной конвертации архива.

In [3]:
import hashlib

processed_path = ROOT / baseline_config.input_path
provenance_path = ROOT / baseline_config.provenance_path
dataset_config_path = ROOT / "configs/dataset.json"
input_candidates = (
    processed_path, provenance_path, dataset_config_path,
    ROOT / "configs/train_baseline.json",
    ROOT / "data/raw/FinancialPhraseBank-v1.0.zip",
    ROOT / "data/interim/financial_phrasebank_75agree.csv",
)
input_checksums_before = {
    path: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in input_candidates if path.is_file()
}
news = load_processed_news(processed_path, provenance_path, dataset_config_path)
news_snapshot = news.copy(deep=True)
print(f"Processed DataFrame shape: {news.shape}")
print(f"Processed CSV SHA-256: {hashlib.sha256(processed_path.read_bytes()).hexdigest()}")
display(news.head())
display(news.dtypes.rename("dtype").to_frame())
class_counts = news["sentiment"].value_counts().reindex(CLASS_ORDER, fill_value=0)
display(pd.DataFrame({"count": class_counts, "share": class_counts / len(news)}))
print(f"Label mapping: {LABEL_MAPPING}")
duplicates = analyze_duplicates(news)
print(f"Raw repeats beyond first: {duplicates.raw_duplicate_rows}")
print(f"Clean repeats beyond first: {duplicates.clean_duplicate_rows}")
print(f"Conflicting groups: {duplicates.has_conflicts}")

Processed DataFrame shape: (3453, 5)
Processed CSV SHA-256: 39dda898a281dc3294be081917b0681524e73edb60f00913183c67714da0c448


,source_row_id,text,sentiment,text_clean,label_id
0,0,"According to Gran , the company has no plans t...",neutral,"according to gran , the company has no plans t...",1
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
2,2,"For the last quarter of 2010 , Componenta 's n...",positive,"for the last quarter of 2010 , componenta 's n...",2
3,3,"In the third quarter of 2010 , net sales incre...",positive,"in the third quarter of 2010 , net sales incre...",2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2


,dtype
source_row_id,int64
text,str
sentiment,str
text_clean,str
label_id,int64


,count,share
sentiment,,
negative,420,0.121633
neutral,2146,0.621489
positive,887,0.256878


Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Raw repeats beyond first: 5
Clean repeats beyond first: 5
Conflicting groups: False


### Проверка результата и интерпретация

Counts описывают входную таблицу, **не качество классификации**.
Большая доля `neutral` объясняет, зачем позже сравнивать macro F1 с accuracy
и с DummyClassifier. Повторяющиеся тексты сохранены и требуют group-aware split:
одна raw/clean группа целиком попадёт в один split в D3-02.

Повторная загрузка ниже должна сохранить значения/порядок DataFrame и bytes
доступных файлов. На D3-01 ещё нет train/validation/test, vocabulary, predictions
и измеренных metrics; готовность окружения не доказывает качество модели.

In [4]:
reloaded_news = load_processed_news(processed_path, provenance_path, dataset_config_path)
pd.testing.assert_frame_equal(news, reloaded_news)
pd.testing.assert_frame_equal(news, news_snapshot)
input_checksums_after = {
    path: hashlib.sha256(path.read_bytes()).hexdigest() for path in input_checksums_before
}
assert input_checksums_after == input_checksums_before
print("Repeated load: identical values, dtypes and row order")
print(f"Input checksums unchanged: {len(input_checksums_before)} files")
print("D3-01: verified input/runtime; split, estimators and fit are pending")

Repeated load: identical values, dtypes and row order
Input checksums unchanged: 6 files
D3-01: verified input/runtime; split, estimators and fit are pending


### Самопроверка

1. Почему `positive=2` не означает «вдвое больше neutral=1»?
2. Почему `text_clean` готов к чтению, но TF-IDF ещё нуждается в `fit`?
3. Какие несоответствия обнаруживает provenance, а какие требуют проверки таблицы?
4. Почему сохранение всех repeats требует группового split?
5. Почему validation_size=0.2 не даёт 20% всей таблицы?
6. Почему counts классов и успешный импорт sklearn ничего не говорят о macro F1?

Следующий пункт — D3-03: построение необученных baseline estimators.

## D3-02 — воспроизводимый stratified group-aware split

**Цель.** Получить manifest `(N, 3)` с `source_row_id, group_id, split`,
три таблицы `(N_split, 5)` и summary в памяти. Вход — проверенный `news` из D3-01.

**Краткая теория.** Train нужен для будущего fit TF-IDF/LogReg, validation —
для сравнения вариантов, held-out test — для единственной итоговой оценки
после заморозки решения в D5-04. Здесь test проверяем только структурно.
Обычный stratified row split может положить повторяющееся предложение в
train и validation: модель встретит тот же текст, и оценка станет слишком лёгкой.
Поэтому единица разделения — группа точного совпадения `text_clean`.
Same-label duplicates сохраняем; fuzzy matching не выполняем.

`group_id` — минимальный `source_row_id`; pandas index не является ID.
Алгоритм `group-stratified-v1`: группы в порядке ID → stratified test=20%,
seed=42 → снова упорядочить outer-train → stratified validation=20% от остатка,
seed=42 → назначить split всем строкам группы, сохраняя исходный порядок.
Это примерно 64/16/20% **групп**; rounding и размеры repeats меняют доли строк.
Нехватка групп/классов и conflicts останавливают split без fallback.

**Вызов кода.** Используем config, зафиксированный до получения ML metrics.


In [5]:
from finnews_sentiment.data.split_data import SPLIT_ORDER, split_processed_news

news_splits = split_processed_news(
    news, test_size=baseline_config.test_size,
    validation_size=baseline_config.validation_size,
    random_state=baseline_config.random_state,
)
split_manifest = news_splits.manifest
train_news = news_splits.splits["train"]
validation_news = news_splits.splits["validation"]
test_news = news_splits.splits["test"]
print(f"Algorithm: {news_splits.summary.algorithm}; seed: {news_splits.summary.random_state}")
print(f"Manifest: {split_manifest.shape}; source rows: {news_splits.summary.rows}; groups: {news_splits.summary.groups}")
split_sizes = pd.DataFrame([
    {"split": name, "rows": stats.rows, "row_share": stats.row_share,
     "groups": stats.groups, "group_share": stats.group_share}
    for name, stats in news_splits.summary.splits.items()
]).set_index("split")
display(split_sizes)
display(split_manifest.head())


Algorithm: group-stratified-v1; seed: 42
Manifest: (3453, 3); source rows: 3453; groups: 3448


,rows,row_share,groups,group_share
split,,,,
train,2210,0.640023,2206,0.639791
validation,553,0.160151,552,0.160093
test,690,0.199826,690,0.200116


,source_row_id,group_id,split
0,0,0,test
1,1,1,train
2,2,2,validation
3,3,3,validation
4,4,4,train


### Counts и доли классов

Для каждой строки таблицы ниже `row_class_share = row_count / N_split`,
а `group_class_share = group_count / G_split`. Stratification распределяет
группы приблизительно по классам; сохранённые repeats меняют веса строк.
Counts test — структурная проверка, не метрики модели и не анализ его ошибок.


In [6]:
split_balance = pd.DataFrame([
    {"split": name, "sentiment": label,
     "row_count": stats.row_class_counts[label],
     "row_class_share": stats.row_class_shares[label],
     "group_count": stats.group_class_counts[label],
     "group_class_share": stats.group_class_shares[label]}
    for name, stats in news_splits.summary.splits.items() for label in CLASS_ORDER
])
display(split_balance.set_index(["split", "sentiment"]))


row_count  row_class_share  group_count  \
split      sentiment                                            
train      negative         269         0.121719          269   
           neutral         1374         0.621719         1370   
           positive         567         0.256561          567   
validation negative          67         0.121157           67   
           neutral          344         0.622061          343   
           positive         142         0.256781          142   
test       negative          84         0.121739           84   
           neutral          428         0.620290          428   
           positive         178         0.257971          178   

                      group_class_share  
split      sentiment                     
train      negative            0.121940  
           neutral             0.621034  
           positive            0.257026  
validation negative            0.121377  
           neutral             0.621377  
           positive            0.257246  
test       negative            0.121739  
           neutral             0.620290  
           positive            0.257971

### Проверка результата

Проверим полное покрытие IDs, отсутствие пересечений по IDs/raw/clean texts,
целостность групп, минимальные group IDs, сохранение порядка и всех строк.
Повторный вызов должен дать тот же manifest и summary, не изменяя inputs.


In [7]:
from itertools import combinations

assert split_manifest["source_row_id"].tolist() == news["source_row_id"].tolist()
assert split_manifest["source_row_id"].is_unique
assert sum(len(frame) for frame in news_splits.splits.values()) == len(news)
assert split_manifest.groupby("group_id")["split"].nunique().eq(1).all()
minimum_ids = news.groupby("text_clean")["source_row_id"].min()
assert split_manifest["group_id"].tolist() == news["text_clean"].map(minimum_ids).tolist()
for left, right in combinations(SPLIT_ORDER, 2):
    for column in ("source_row_id", "text", "text_clean"):
        assert set(news_splits.splits[left][column]).isdisjoint(news_splits.splits[right][column])
for name, frame in news_splits.splits.items():
    assert set(frame["sentiment"]) == set(CLASS_ORDER)
    ids = split_manifest.loc[split_manifest["split"] == name, "source_row_id"]
    pd.testing.assert_frame_equal(frame, news.loc[news["source_row_id"].isin(ids)])
repeated_splits = split_processed_news(
    news, test_size=baseline_config.test_size,
    validation_size=baseline_config.validation_size,
    random_state=baseline_config.random_state,
)
pd.testing.assert_frame_equal(split_manifest, repeated_splits.manifest)
assert news_splits.summary == repeated_splits.summary
pd.testing.assert_frame_equal(news, news_snapshot)
assert input_checksums_before == {
    path: hashlib.sha256(path.read_bytes()).hexdigest() for path in input_checksums_before
}
print("Verified: complete coverage, intact groups, disjoint IDs/raw/clean texts, original row order")
print("Repeated split identical; source DataFrame and input bytes unchanged")
print(f"Disjointness: {news_splits.summary.disjoint}")


Verified: complete coverage, intact groups, disjoint IDs/raw/clean texts, original row order
Repeated split identical; source DataFrame and input bytes unchanged
Disjointness: {'source_row_id': True, 'text': True, 'text_clean': True}


### Интерпретация и ограничения

Фактические размеры train/validation/test: **2210/553/690 строк**,
**2206/552/690 групп**. Всего 3453 строки и 3448 групп: все 5 repeats сохранены.
Четыре repeat rows оказались в train, одна — в validation. Разницы между долями
строк и групп небольшие, но это результат данных, а не гарантия алгоритма.
Во всех splits три класса, neutral остаётся большинством; точных пересечений нет.

Группы защищают от leakage через точные совпадения, но не через парафразы.
Одинаковые dataset/config/runtime воспроизводят manifest; другой dataset или
версия sklearn могут изменить назначение даже при том же seed.
Manifest и summary пока в памяти; сохранение reference artifacts — D3-06.

### Самопроверка

1. Почему stratify по строкам недостаточно при same-label duplicates?
2. Чем число групп отличается от числа строк, и к чему относятся 64/16/20%?
3. Почему outer-train нужно снова сортировать перед вторым split?
4. Что означает `group_id`, и почему это не pandas index?
5. Почему class counts test сейчас разрешены, а test metrics отложены?
6. Почему split не должен молча менять seed при недостатке групп?

Следующий пункт — D3-03: создание необученных estimators.


## D3-03 — builders TF-IDF/LogReg и Dummy baseline

**Цель.** Из проверенного `baseline_config` построить новые необученные `Pipeline` и `DummyClassifier`.

**Краткая теория.** `TfidfVectorizer` строит unigram/bigram vocabulary и IDF при будущем `fit` только по train, затем выдаёт sparse матрицу `(N_train, V)`, `V <= 5000`. `Pipeline` связывает fit/transform в одном объекте: при обучении на train validation пройдёт через те же параметры vocabulary/IDF. LogReg учит классификатор на этих sparse признаках. Dummy предскажет частый класс, обучаясь по train labels и принимая constant feature array `(N, 1)`.

Параметры берутся из config, закреплённого до validation metrics. Вызов ниже только создаёт estimators: `fit` и `predict` не вызываются.


In [8]:
from finnews_sentiment.models.train_model import build_baseline_pipeline, build_dummy_classifier

baseline_pipeline = build_baseline_pipeline(baseline_config)
dummy_classifier = build_dummy_classifier(baseline_config)
display(pd.Series(baseline_pipeline.get_params(deep=False), name="Pipeline top-level params").to_frame())
display(pd.Series(baseline_pipeline.named_steps["tfidf"].get_params(deep=False), name="TF-IDF settings").to_frame())
display(pd.Series(baseline_pipeline.named_steps["clf"].get_params(deep=False), name="LogReg settings").to_frame())
print(f"Pipeline steps: {list(baseline_pipeline.named_steps)}")
print(f"Dummy strategy: {dummy_classifier.get_params(deep=False)["strategy"]}")
assert list(baseline_pipeline.named_steps) == ["tfidf", "clf"]
assert not hasattr(baseline_pipeline.named_steps["tfidf"], "vocabulary_")
assert not hasattr(baseline_pipeline.named_steps["clf"], "classes_")
assert not hasattr(dummy_classifier, "classes_")
print("Builders returned unfitted estimators; no fit or prediction performed")

,Pipeline top-level params
memory,None
steps,"[(tfidf, TfidfVectorizer(lowercase=False, max_..."
transform_input,None
verbose,False


,TF-IDF settings
analyzer,word
binary,False
decode_error,strict
dtype,<class 'numpy.float64'>
encoding,utf-8
input,content
lowercase,False
max_df,1.0
max_features,5000
min_df,1


,LogReg settings
C,1.0
class_weight,None
dual,False
fit_intercept,True
intercept_scaling,1
l1_ratio,0.0
max_iter,1000
n_jobs,None
penalty,deprecated
random_state,42


Pipeline steps: ['tfidf', 'clf']
Dummy strategy: most_frequent
Builders returned unfitted estimators; no fit or prediction performed


### Интерпретация и ограничения

TF-IDF `max_features=5000` — верхняя граница `V`, а не обещание ровно 5000 признаков. Pipeline сохраняет sparse представление и fit boundary. Bag-of-n-grams теряет порядок за пределами соседних 1–2 слов и не моделирует контекст. Стандартный tokenizer `(?u)\b\w\w+\b` не создаёт отдельные признаки для `%` и валютной пунктуации, пропускает односимвольные tokens и может разбивать `n't`; знаки, оставленные preprocessing, могут не войти в vocabulary как отдельные признаки. Custom tokenizer здесь не добавляем.

Dummy — ориентир majority-class поведения; constant features он получит вместе с train в D3-04. Сейчас у обоих estimators нет learned attributes.

### Самопроверка

1. Почему Pipeline защищает границу между train fit и validation transform?
2. Что теряет bag-of-n-grams?
3. Почему `%`, символ валюты и `n't` не обязательно станут отдельными TF-IDF признаками?
4. Почему `max_features=5000` не означает ровно 5000 столбцов?

Следующий пункт — D3-04: обучение только на train и predictions для validation. На этом checkpoint обучение не выполнялось.

## D3-04 — fit только на train и validation predictions

**Цель.** Обучить новый TF-IDF/LogReg Pipeline и DummyClassifier на `train_news`, затем получить predictions для тех же validation rows с привязкой к `source_row_id`. Вход — проверенные splits D3-02 и фиксированный config. Test проверяется структурно, но не передаётся vectorizer или модели.

**Краткая теория.** `fit_transform(train.text_clean)` обучает vocabulary/IDF и LogReg параметры на train. Validation проходит через уже обученный `transform` внутри `predict`: словарь и IDF не обновляются. Dummy получает constant feature array `(N_train, 1)` и учит частый train class. Sparse TF-IDF имеет форму `(N_train, V)`; в dense формат её не преобразуем.

Любой `ConvergenceWarning` переводится в явную ошибку checkpoint. Параметры здесь не подбираются по validation.

In [9]:
import numpy as np
from finnews_sentiment.models.train_model import fit_baselines

baseline_fit = fit_baselines(news_splits, baseline_config)
validation_predictions = baseline_fit.validation_predictions
vectorizer_fitted = baseline_fit.pipeline.named_steps["tfidf"]
train_feature_shape = baseline_fit.train_feature_shape
print(f"Train sparse TF-IDF shape: {train_feature_shape}")
print(f"Vocabulary features V: {baseline_fit.feature_count} (config max_features={baseline_config.tfidf['max_features']})")
print(f"LogReg classes_: {baseline_fit.pipeline.named_steps['clf'].classes_.tolist()}")
print(f"Dummy classes_: {list(baseline_fit.dummy.classes_)}")
print(f"Convergence: {baseline_fit.convergence_status}")
print(f"Validation predictions shape: {validation_predictions.shape}")
display(validation_predictions.head())
assert train_feature_shape[0] == len(train_news)
assert train_feature_shape[1] <= baseline_config.tfidf["max_features"]
assert validation_predictions["source_row_id"].tolist() == validation_news["source_row_id"].tolist()
assert len(validation_predictions) == len(validation_news)
assert set(validation_predictions["baseline_prediction"]).issubset({0, 1, 2})
assert set(validation_predictions["dummy_prediction"]).issubset({0, 1, 2})
assert baseline_fit.convergence_status == "converged_no_warning"

Train sparse TF-IDF shape: (2210, 5000)
Vocabulary features V: 5000 (config max_features=5000)
LogReg classes_: [0, 1, 2]
Dummy classes_: [np.int64(0), np.int64(1), np.int64(2)]
Convergence: converged_no_warning
Validation predictions shape: (553, 3)


,source_row_id,baseline_prediction,dummy_prediction
0,2,2,1
1,3,2,1
2,11,2,1
3,16,1,1
4,28,1,1


### Проверка fit boundary и alignment

Уникальные validation-only terms — примеры словосочетаний, которых не было в train. Они не могут попасть в train vocabulary. Повторный `predict` должен оставить vocabulary и IDF без изменений. `source_row_id` в predictions должен точно совпасть с порядком validation table; сами метрики и ошибки классификации будут разбираться в D3-05.


In [10]:
analyzer = vectorizer_fitted.build_analyzer()
train_terms = {term for text in train_news[baseline_config.text_col] for term in analyzer(text)}
validation_terms = {term for text in validation_news[baseline_config.text_col] for term in analyzer(text)}
validation_only_terms = validation_terms - train_terms
vocabulary_before_predict = dict(vectorizer_fitted.vocabulary_)
idf_before_predict = vectorizer_fitted.idf_.copy()
repeated_predictions = baseline_fit.pipeline.predict(validation_news[baseline_config.text_col].tolist())
assert validation_only_terms
assert validation_only_terms.isdisjoint(vocabulary_before_predict)
assert vectorizer_fitted.vocabulary_ == vocabulary_before_predict
np.testing.assert_array_equal(vectorizer_fitted.idf_, idf_before_predict)
np.testing.assert_array_equal(repeated_predictions, validation_predictions["baseline_prediction"].to_numpy())
assert list(baseline_fit.pipeline.named_steps["clf"].classes_) == [0, 1, 2]
assert tuple(baseline_fit.dummy.classes_) == (0, 1, 2)
print(f"Validation-only TF-IDF terms excluded from vocabulary: {len(validation_only_terms)}")
print("Vocabulary/IDF unchanged; predictions remain aligned and reproducible")
print("Test split: structural checks only; no transform/predict/score")

Validation-only TF-IDF terms excluded from vocabulary: 5706
Vocabulary/IDF unchanged; predictions remain aligned and reproducible
Test split: structural checks only; no transform/predict/score


### Интерпретация и самопроверка

В отчёте `(N_train, V)` — sparse train matrix; `V` ограничено `max_features`, но может быть меньше. `classes_=[0,1,2]` соответствует `negative, neutral, positive`. Dummy выдаёт baseline majority class, рассчитанный только по train. Validation predictions — массивы длины `N_valid`, прикреплённые к исходным IDs; они сами по себе не являются оценкой качества.

1. Что обучается в TF-IDF и что обучается в LogReg?
2. Чем `transform` отличается от `fit_transform`?
3. Почему vocabulary/IDF нельзя дообучать на validation?
4. Почему Dummy получает постоянный feature, хотя его стратегия основана на частотах labels?

Следующий пункт — D3-05: метрики baseline и Dummy на одних и тех же validation rows. Test остаётся held-out.

## D3-05 — validation metrics и сравнение с Dummy

**Цель.** Оценить baseline и Dummy на одних и тех же `N_valid` строках. Вход — true `label_id` из validation и aligned predictions D3-04; test не используется.

**Краткая теория.** Accuracy — доля верных предсказаний среди всех строк. Macro F1 сначала считает F1 отдельно для `negative`, `neutral`, `positive`, затем усредняет классы с одинаковым весом. Поэтому при доминировании neutral accuracy может выглядеть приемлемо, даже если редкие классы распознаются плохо. Per-class precision/recall/F1/support объясняют это различие; `zero_division=0` сохраняет отсутствующие predictions класса в таблице с нулевыми метриками.

Это оценка одного group-aware validation split. Она не доказывает устойчивость на новых периодах или live финансовых новостях. Параметры не меняются по результатам.

In [11]:
from finnews_sentiment.models.evaluate import evaluate_baselines

assert validation_predictions["source_row_id"].tolist() == validation_news["source_row_id"].tolist()
evaluation = evaluate_baselines(
    validation_news[baseline_config.target_col].to_numpy(dtype=np.int64),
    validation_predictions["baseline_prediction"].to_numpy(dtype=np.int64),
    validation_predictions["dummy_prediction"].to_numpy(dtype=np.int64),
)
aggregate_metrics = evaluation.comparison[["model", "n_samples", "accuracy", "macro_f1"]].set_index("model")
display(aggregate_metrics)
per_class_rows = []
for model_name, metrics in (("baseline", evaluation.baseline), ("dummy", evaluation.dummy)):
    for class_metrics in metrics.per_class:
        per_class_rows.append({
            "model": model_name, "class": class_metrics.class_name,
            "precision": class_metrics.precision, "recall": class_metrics.recall,
            "f1": class_metrics.f1, "support": class_metrics.support,
        })
per_class_metrics = pd.DataFrame(per_class_rows).set_index(["model", "class"])
display(per_class_metrics)
assert evaluation.baseline.n_samples == evaluation.dummy.n_samples == len(validation_news)
assert sum(item.support for item in evaluation.baseline.per_class) == len(validation_news)
assert sum(item.support for item in evaluation.dummy.per_class) == len(validation_news)
assert [item.class_name for item in evaluation.baseline.per_class] == list(CLASS_ORDER)
assert [item.class_name for item in evaluation.dummy.per_class] == list(CLASS_ORDER)
print(f"Both metric sets use the same {len(validation_news)} validation rows; test was not evaluated")

,n_samples,accuracy,macro_f1
model,,,
baseline,553,0.813743,0.720965
dummy,553,0.622061,0.255667


precision    recall        f1  support
model    class                                           
baseline negative   0.861111  0.462687  0.601942       67
         neutral    0.815534  0.976744  0.888889      344
         positive   0.790476  0.584507  0.672065      142
dummy    negative   0.000000  0.000000  0.000000       67
         neutral    0.622061  1.000000  0.767001      344
         positive   0.000000  0.000000  0.000000      142

Both metric sets use the same 553 validation rows; test was not evaluated


### Интерпретация и самопроверка

Сравни aggregate macro F1 и accuracy, затем per-class recall/support. Dummy задаёт majority-class reference, а не целевую метрику; победа baseline не является условием приёмки. Macro F1 может быть низким при высокой accuracy, когда частый neutral class распознаётся хорошо, а редкие классы — слабее. Нулевой recall означает, что среди реальных строк этого класса модель не предсказала ни одной верно. Row metrics сохраняют вес повторов данных; group split снижает риск exact-text leakage, но один validation split не показывает статистическую устойчивость. На этих результатах нельзя настраивать параметры или делать выводы о test/live качестве.

1. Почему macro F1 может быть заметно ниже accuracy?
2. Что означает нулевой recall класса, если его support ненулевой?
3. Почему comparison справедлив только при одинаковых validation source IDs?
4. Какие выводы одного split остаются ненадёжными?

Следующий пункт — D3-06: сохранить baseline reference и provenance для воспроизведения. Confusion matrix и разбор отдельных ошибок остаются Day05.

## D3-06 — baseline reference artifacts и offline runner

**Цель.** Запустить единый workflow D3-01…D3-05 и сохранить split manifest, summary, validation predictions/metrics, effective config и provenance. Notebook вызывает тот же `run_baseline`, что и CLI; отдельного обучения или вычисления метрик в notebook нет.

**Краткая теория.** Artifact reference фиксирует данные, параметры и результаты, чтобы Day04 мог сравнивать новые эксперименты с baseline. Provenance связывает отчёты с входными checksums, Day02 policy, seed, split algorithm, vocabulary/classes и версиями runtime. Test остаётся только структурно включённым в manifest: test predictions/metrics и inference bundle не создаются.

Offline команда из корня: `PYTHONPATH=src venv/bin/python -m finnews_sentiment.models.train_model --config configs/train_baseline.json`. Download не выполняется.

In [12]:
from finnews_sentiment.models.run_baseline import run_baseline

baseline_run = run_baseline(ROOT / "configs/train_baseline.json", ROOT)
artifact_paths = pd.Series({
    "splits": str(baseline_run.artifacts.splits.relative_to(ROOT)),
    "split_summary": str(baseline_run.artifacts.split_summary.relative_to(ROOT)),
    "validation_predictions": str(baseline_run.artifacts.validation_predictions.relative_to(ROOT)),
    "metrics": str(baseline_run.artifacts.metrics.relative_to(ROOT)),
    "config": str(baseline_run.artifacts.config.relative_to(ROOT)),
    "provenance": str(baseline_run.artifacts.provenance.relative_to(ROOT)),
}, name="path")
display(artifact_paths.to_frame())
saved_manifest = pd.read_csv(
    baseline_run.artifacts.splits, dtype={"source_row_id": "int64", "group_id": "int64"},
)
pd.testing.assert_frame_equal(saved_manifest, news_splits.manifest)
saved_predictions = pd.read_csv(baseline_run.artifacts.validation_predictions, dtype="int64")
expected_saved_predictions = pd.DataFrame({
    "source_row_id": validation_news["source_row_id"].to_numpy(),
    "true_label_id": validation_news[baseline_config.target_col].to_numpy(),
    "baseline_pred_label_id": validation_predictions["baseline_prediction"].to_numpy(),
    "dummy_pred_label_id": validation_predictions["dummy_prediction"].to_numpy(),
})
pd.testing.assert_frame_equal(saved_predictions, expected_saved_predictions)
saved_metrics = json.loads(baseline_run.artifacts.metrics.read_text(encoding="utf-8"))
saved_summary = json.loads(baseline_run.artifacts.split_summary.read_text(encoding="utf-8"))
saved_provenance = json.loads(baseline_run.artifacts.provenance.read_text(encoding="utf-8"))
assert saved_metrics["evaluation_split"] == "validation"
assert saved_metrics["n_valid"] == evaluation.baseline.n_samples
assert saved_metrics["baseline"]["macro_f1"] == evaluation.baseline.macro_f1
assert saved_metrics["dummy"]["accuracy"] == evaluation.dummy.accuracy
assert saved_summary["algorithm"] == news_splits.summary.algorithm
assert saved_provenance["checksums"]["predictions"] == hashlib.sha256(baseline_run.artifacts.validation_predictions.read_bytes()).hexdigest()
assert saved_provenance["checksums"]["metrics"] == hashlib.sha256(baseline_run.artifacts.metrics.read_bytes()).hexdigest()
print(f"Round-trip verified: manifest rows={len(saved_manifest)}, validation predictions={len(saved_predictions)}")
print(f"Evaluation split={saved_metrics['evaluation_split']}; no test predictions or metrics saved")

,path
splits,data/processed/day03/splits.csv
split_summary,reports/day03/split_summary.json
validation_predictions,reports/day03/validation_predictions.csv
metrics,reports/day03/metrics.json
config,reports/day03/config.json
provenance,reports/day03/provenance.json


Round-trip verified: manifest rows=3453, validation predictions=553
Evaluation split=validation; no test predictions or metrics saved


### Результат и самопроверка

Проверь paths и provenance: сохранённые validation IDs должны совпадать с D3-02/D3-04, а metrics — с D3-05. Все шесть outputs локальные и игнорируются Git. Checksums позволяют проверить связь reference с input bytes и отчётами. Binary model file не сохраняется; воспроизводимость обещается для splits/predictions/metrics в том же runtime, не для binary model bytes между платформами. Сохранение набора файлов не является одной общей транзакцией; при I/O сбое команду можно повторить.

1. Почему provenance нужно связывать с dataset revision, policy и checksum?
2. Почему matching seed не гарантирует matching split при изменении данных/runtime?
3. Какие артефакты нужны для baseline reference, а какие относятся к inference bundle?

Следующий пункт — D3-07: итоговая интерпретация и приёмка Day03.

## D3-07 — выводы и итоговая приёмка Day03

**Цель.** Зафиксировать baseline reference для Day04 и проверить, что notebook-результаты согласованы с локальными артефактами.

**Краткая теория.** Модель можно сравнивать с baseline только при неизменной границе данных и понятной метрике. Validation/CV предназначены для выбора; held-out test нужен для отдельной финальной оценки и сейчас не используется. Один split и лексические признаки TF-IDF не доказывают качество на новых финансовых новостях.

D3-01…D3-06 создали reusable pipeline, train-only fit, aligned validation predictions/metrics и provenance. Следующая ячейка сводит проверяемые факты, не обучая модель повторно и не обращаясь к test labels для оценки.

In [13]:
acceptance_checks = pd.Series({
    "three split manifest round-trip": saved_manifest.equals(news_splits.manifest),
    "source IDs, raw text and clean text disjoint": all(saved_summary["disjoint"].values()),
    "validation predictions aligned": saved_predictions["source_row_id"].equals(validation_news["source_row_id"].reset_index(drop=True)),
    "validation metrics match D3-05": saved_metrics["n_valid"] == len(validation_news) and saved_metrics["baseline"]["macro_f1"] == evaluation.baseline.macro_f1 and saved_metrics["dummy"]["macro_f1"] == evaluation.dummy.macro_f1,
    "provenance covers current metrics": saved_provenance["checksums"]["metrics"] == hashlib.sha256(baseline_run.artifacts.metrics.read_bytes()).hexdigest(),
    "held-out test has no evaluation metrics": saved_metrics["evaluation_split"] == "validation" and "test" not in saved_metrics,
}, name="passed")
assert acceptance_checks.all()
display(acceptance_checks.to_frame())
print(f"Baseline: accuracy={saved_metrics['baseline']['accuracy']:.6f}, macro F1={saved_metrics['baseline']['macro_f1']:.6f}")
print(f"Dummy: accuracy={saved_metrics['dummy']['accuracy']:.6f}, macro F1={saved_metrics['dummy']['macro_f1']:.6f}")
print("Test was not evaluated; do not use this single validation split as a live-news quality guarantee.")

,passed
three split manifest round-trip,True
"source IDs, raw text and clean text disjoint",True
validation predictions aligned,True
validation metrics match D3-05,True
provenance covers current metrics,True
held-out test has no evaluation metrics,True


Baseline: accuracy=0.813743, macro F1=0.720965
Dummy: accuracy=0.622061, macro F1=0.255667
Test was not evaluated; do not use this single validation split as a live-news quality guarantee.


### Результат, интерпретация и самопроверка

Train/validation/test содержат 2210/553/690 строк и 2206/552/690 групп; пять exact repeats оставлены и распределены только между train и validation. Train TF-IDF shape `(2210, 5000)`. На validation baseline accuracy/macro F1 — 0.813743/0.720965, Dummy — 0.622061/0.255667. Dummy всегда выбирает majority class `neutral`, поэтому его accuracy скрывает нулевой recall двух остальных классов; macro F1 показывает их вклад. Подробный class balance и ограничения записаны в `reports/day03/observations.md`, команды запуска — в `docs/day03.md`.

Это baseline одной выборки; TF-IDF word n-grams не моделируют дальний контекст, семантику или временной сдвиг. Значения не являются гарантией качества на новых новостях. Test не проходил fit/predict/score и метрики по нему не считались.

1. На каких данных Day04 допустимо выбирать модель и почему?
2. Почему использование test для настройки меняет смысл итоговой оценки?
3. Что baseline не доказывает о качестве на новых финансовых новостях?